# Layers of defence for an agent

An ordinary, unprivileged pod stands in for an unmanaged agent. It makes normal HTTP and MCP requests.
For each boundary, try the request, add a policy, then repeat it and read the evidence.
The model is a local echo simulator. The tool named delete_records only returns a dry-run message.
No real personal data or model credentials are used.

This lab uses Enterprise agentgateway 2026.8.2 for its global per-identity rate limit.
Istio's trust domain is mesh1. Kindnet does not enforce NetworkPolicy on this cluster,
so the egress chapter controls example.com through a namespace-scoped waypoint. It does not claim
to close every internet path. The content chapter masks rather than refuses; detection observes rather than blocks.

## Connect

Set the environment when following the notebook directly. The console sets it for every step.

In [ ]:
export DD=demo-scripts/defence DD_URL=http://dd-gateway.dd-gateway.svc
export DD_STATE="${TMPDIR:-/tmp}/defence-lab"
. "$DD/helpers.sh"

## 1. Create the lab's workloads

Create four namespaces on mesh1, an unprivileged stand-in agent, a local echo model, two harmless MCP tools and a dedicated gateway.

All demonstration traffic comes from unmanaged-agent. Reset removes these workloads. The installed platform is shared.

### Create the namespaces

Enrol this lab's namespaces in ambient.

**Expected:** Four dd- namespaces are created.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: v1
kind: Namespace
metadata:
  name: dd-agents
  labels: {istio.io/dataplane-mode: ambient}
---
apiVersion: v1
kind: Namespace
metadata:
  name: dd-models
  labels: {istio.io/dataplane-mode: ambient}
---
apiVersion: v1
kind: Namespace
metadata:
  name: dd-tools
  labels: {istio.io/dataplane-mode: ambient}
---
apiVersion: v1
kind: Namespace
metadata:
  name: dd-gateway
  labels: {istio.io/dataplane-mode: ambient}
YAML

### Load the sample model and tools

Store the model and tool server's Python files in Kubernetes ConfigMaps. The next step mounts these files into pods and runs them.

**Expected:** Both ConfigMaps are created with the Python files the pods will run.

In [ ]:
kubectl --context kind-mesh1 -n dd-models create configmap model-code --from-file="$DD/model.py" --dry-run=client -o yaml | kubectl --context kind-mesh1 apply -f -
kubectl --context kind-mesh1 -n dd-tools create configmap tools-code --from-file="$DD/tools.py" --dry-run=client -o yaml | kubectl --context kind-mesh1 apply -f -

### Start the agent, model and tools

Start the agent pod, model simulator and tool server using container images already on mesh1.

**Expected:** The agent is Ready and both Deployments roll out.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: v1
kind: ServiceAccount
metadata: {name: unmanaged-agent, namespace: dd-agents}
automountServiceAccountToken: false
---
apiVersion: v1
kind: Pod
metadata:
  name: unmanaged-agent
  namespace: dd-agents
  labels: {app: unmanaged-agent}
spec:
  serviceAccountName: unmanaged-agent
  automountServiceAccountToken: false
  securityContext: {runAsNonRoot: true, runAsUser: 1000}
  containers:
  - name: curl
    image: curlimages/curl:8.12.1
    command: [sleep, infinity]
    securityContext:
      allowPrivilegeEscalation: false
      capabilities: {drop: [ALL]}
    resources:
      requests: {cpu: 5m, memory: 8Mi}
      limits: {cpu: 100m, memory: 32Mi}
---
apiVersion: apps/v1
kind: Deployment
metadata: {name: model, namespace: dd-models}
spec:
  replicas: 1
  selector: {matchLabels: {app: dd-model}}
  template:
    metadata: {labels: {app: dd-model}}
    spec:
      automountServiceAccountToken: false
      containers:
      - name: model
        image: localhost:5001/llm-gateway-mock:latest
        imagePullPolicy: IfNotPresent
        command: [python, -u, /lab/model.py]
        ports: [{containerPort: 8000}]
        readinessProbe: {httpGet: {path: /health, port: 8000}, periodSeconds: 2}
        resources:
          requests: {cpu: 5m, memory: 16Mi}
          limits: {cpu: 100m, memory: 64Mi}
        volumeMounts: [{name: code, mountPath: /lab, readOnly: true}]
      volumes: [{name: code, configMap: {name: model-code}}]
---
apiVersion: v1
kind: Service
metadata: {name: model, namespace: dd-models}
spec:
  selector: {app: dd-model}
  ports: [{name: http, port: 8000, targetPort: 8000}]
---
apiVersion: apps/v1
kind: Deployment
metadata: {name: tools, namespace: dd-tools}
spec:
  replicas: 1
  selector: {matchLabels: {app: dd-tools}}
  template:
    metadata: {labels: {app: dd-tools}}
    spec:
      automountServiceAccountToken: false
      containers:
      - name: tools
        image: localhost:5001/calc-mcp:latest
        imagePullPolicy: IfNotPresent
        command: [python, -u, /lab/tools.py]
        ports: [{containerPort: 3000}]
        readinessProbe: {tcpSocket: {port: 3000}, periodSeconds: 2}
        resources:
          requests: {cpu: 5m, memory: 32Mi}
          limits: {cpu: 100m, memory: 128Mi}
        volumeMounts: [{name: code, mountPath: /lab, readOnly: true}]
      volumes: [{name: code, configMap: {name: tools-code}}]
---
apiVersion: v1
kind: Service
metadata: {name: tools, namespace: dd-tools}
spec:
  selector: {app: dd-tools}
  ports: [{name: http, port: 3000, targetPort: 3000}]
YAML
kubectl --context kind-mesh1 -n dd-agents wait pod/unmanaged-agent --for=condition=Ready --timeout=120s
kubectl --context kind-mesh1 -n dd-models rollout status deploy/model --timeout=120s
kubectl --context kind-mesh1 -n dd-tools rollout status deploy/tools --timeout=120s

### Create the model and MCP gateway

Give this lab its own gateway, model backend and MCP route.

**Expected:** dd-gateway is Ready.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: {name: dd-gateway, namespace: dd-gateway}
spec:
  gatewayClassName: enterprise-agentgateway
  listeners:
  - name: http
    protocol: HTTP
    port: 80
    allowedRoutes: {namespaces: {from: Same}}
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: {name: model, namespace: dd-gateway}
spec:
  ai:
    provider:
      openai: {model: dd-echo}
      host: model.dd-models.svc.cluster.local
      port: 8000
      path: /v1/chat/completions
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayBackend
metadata: {name: tools, namespace: dd-gateway}
spec:
  mcp:
    targets:
    - name: records
      static: {host: tools.dd-tools.svc.cluster.local, port: 3000, path: /mcp}
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: model, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: PathPrefix, value: /v1/chat/completions}}]
    backendRefs:
    - {name: model, group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend}
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: {name: tools, namespace: dd-gateway}
spec:
  parentRefs: [{name: dd-gateway}]
  rules:
  - matches: [{path: {type: PathPrefix, value: /mcp}}]
    backendRefs:
    - {name: tools, group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend}
YAML
kubectl --context kind-mesh1 -n dd-gateway wait gateway/dd-gateway --for=condition=Programmed --timeout=120s
kubectl --context kind-mesh1 -n dd-gateway rollout status deploy/dd-gateway --timeout=120s

### Declare the outside destination

Put example.com behind a waypoint visible only to dd-agents. No deny policy is attached yet.

**Expected:** The waypoint is Ready and an ordinary HTTPS request still returns 200.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
# Only this namespace sees the ServiceEntry. Other labs keep their DNS answers.
# This covers example.com, not arbitrary internet destinations or IP literals.
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: dd-egress
  namespace: dd-agents
  labels: {istio.io/waypoint-for: service}
spec:
  gatewayClassName: istio-waypoint
  listeners: [{name: mesh, port: 15008, protocol: HBONE}]
---
apiVersion: networking.istio.io/v1
kind: ServiceEntry
metadata:
  name: outside-example
  namespace: dd-agents
  labels: {istio.io/use-waypoint: dd-egress}
spec:
  exportTo: ["."]
  hosts: [example.com]
  ports: [{number: 443, name: tls, protocol: TLS}]
  resolution: DNS
  location: MESH_EXTERNAL
YAML
kubectl --context kind-mesh1 -n dd-agents rollout status deploy/dd-egress --timeout=120s
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 --retry 10 --retry-all-errors --retry-delay 1 -o /dev/null -w 'Outside HTTP %{http_code}\n' https://example.com

### Prepare test identities

Generate a lab-local signing key and public JWT policy. The policy is not applied yet.

**Expected:** The public JWT policy is ready. The private key stays local and Reset removes it.

In [ ]:
python3 "$DD/identity.py" setup

### Select the installed rate limiter

Create a lab-owned Service pointing to the main rate-limiter ReplicaSet. On this rig the default Service also selects the waypoint limiter, whose separate counter would double the allowance.

**Expected:** dd-rate-limiter selects exactly the main limiter. No platform Deployment is changed.

In [ ]:
HASH=$(kubectl --context kind-mesh1 -n agentgateway-system get rs -o json | jq -er '[.items[] | select(any(.metadata.ownerReferences[]?; .kind == "Deployment" and .name == "rate-limiter-enterprise-agentgateway")) | select(.spec.replicas > 0)] | sort_by(.metadata.creationTimestamp) | last | .metadata.labels["pod-template-hash"]')
jq -n --arg hash "$HASH" '{apiVersion:"v1",kind:"Service",metadata:{name:"dd-rate-limiter",namespace:"agentgateway-system"},spec:{selector:{app:"rate-limiter","pod-template-hash":$hash},ports:[{name:"grpc",port:8083,targetPort:8083}]}}' | kubectl --context kind-mesh1 apply -f -
kubectl --context kind-mesh1 -n agentgateway-system get service dd-rate-limiter -o yaml

## 2. Control access to an outside destination

The pod can currently reach example.com. Deny traffic at its egress waypoint, then repeat the same HTTPS request.

This is destination-scoped egress control. Kindnet accepts but does not enforce NetworkPolicy. Other destinations are outside this chapter's claim.

### Without the control

Fetch example.com through the open waypoint.

**Expected:** The HTTPS request returns 200.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -o /dev/null -w 'Outside HTTP %{http_code}\n' https://example.com

### Add the control

Deny all traffic handled by this lab's egress waypoint.

**Expected:** The AuthorizationPolicy is applied.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: {name: outside-deny, namespace: dd-agents}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: Gateway, name: dd-egress}
  action: DENY
  rules: [{}]
YAML
sleep 2

### With the control

Repeat the HTTPS request and read the waypoint's TCP RBAC counter. A curl failure alone would not prove a policy refusal.

**Expected:** The connection is refused and tcp.rbac.denied is greater than zero.

In [ ]:
if kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -o /dev/null https://example.com; then printf 'Unexpected outside access\n'; exit 1; else printf 'Outside request refused\n'; fi
kubectl --context kind-mesh1 -n dd-agents exec deploy/dd-egress -- pilot-agent request GET 'stats?filter=rbac'

## 3. Allow only the gateway to reach the model

An ambient certificate identifies the pod, but no policy yet prevents it from calling the model directly. Add the gateway-only rule to models and tools.

The platform checks the caller's SPIFFE identity. User JWTs do not replace this boundary.

### Without the control

Call the model service directly from the agent pod.

**Expected:** The model echoes the request with HTTP 200.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -w '\nHTTP %{http_code}\n' http://model.dd-models.svc:8000/v1/chat/completions \
  -H 'Content-Type: application/json' -d '{"model":"dd-echo","messages":[{"role":"user","content":"ordinary model request"}]}'

### Add the control

Allow only mesh1/ns/dd-gateway/sa/dd-gateway into the model and tool namespaces.

**Expected:** Both gateway-only policies are applied.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: {name: gateway-only, namespace: dd-models}
spec:
  action: ALLOW
  rules:
  - from:
    - source:
        principals: [mesh1/ns/dd-gateway/sa/dd-gateway]
---
# The same boundary protects tools from callers that bypass the MCP gateway.
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: {name: gateway-only, namespace: dd-tools}
spec:
  action: ALLOW
  rules:
  - from:
    - source:
        principals: [mesh1/ns/dd-gateway/sa/dd-gateway]
YAML
sleep 2

### With the control

Repeat the direct call, then use the gateway. Read the inbound ztunnel refusal for this workload.

**Expected:** Direct access is refused, the gateway still returns 200, and the log names unmanaged-agent.

In [ ]:
if kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 http://model.dd-models.svc:8000/v1/chat/completions -H 'Content-Type: application/json' -d '{"model":"dd-echo","messages":[{"role":"user","content":"ordinary model request"}]}'; then exit 1; else printf 'Direct model request refused\n'; fi
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" \
  -H 'Content-Type: application/json' -d '{"model":"dd-echo","messages":[{"role":"user","content":"ordinary model request"}]}'
kubectl --context kind-mesh1 -n istio-system logs -l app=ztunnel --since=30s --tail=3000 | jq -c 'select(.["src.namespace"] == "dd-agents" and .direction == "inbound" and (.error // "" | contains("policy rejection"))) | {source:.["src.identity"], destination:.["dst.namespace"], error}'

## 4. Require a signed user token

The gateway is an allowed workload. Now require the person or application using it to present a token from the lab's issuer, with the right audience.

The same pod gets 401 without a token and 200 with Alice's token. These are different identities at different boundaries.

### Without the control

Call the model through the gateway without a token.

**Expected:** HTTP 200, because caller authentication is not configured yet.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" \
  -H 'Content-Type: application/json' -d '{"model":"dd-echo","messages":[{"role":"user","content":"ordinary model request"}]}'

### Add the control

Read the entire public JWT policy, then apply Strict authentication to the lab gateway.

**Expected:** The policy is Accepted and Attached.

In [ ]:
cat "$DD_STATE/07-caller-identity.json"
kubectl --context kind-mesh1 apply -f "$DD_STATE/07-caller-identity.json"
dd_wait_policy caller-identity

### With the control

Repeat the anonymous request, then present a freshly signed Alice token. Read the gateway's refusal log.

**Expected:** 401 with no token, 200 with Alice's token, and a recorded authentication failure.

In [ ]:
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" \
  -H 'Content-Type: application/json' -d '{"model":"dd-echo","messages":[{"role":"user","content":"ordinary model request"}]}'
ALICE=$(python3 "$DD/identity.py" token --user alice)
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" -H "Authorization: Bearer $ALICE" \
  -H 'Content-Type: application/json' -d '{"model":"dd-echo","messages":[{"role":"user","content":"ordinary model request"}]}'
kubectl --context kind-mesh1 -n dd-gateway logs deploy/dd-gateway --since=15s | grep 'http.status=401'

## 5. Give Alice only the read tool

The MCP server offers read_status and delete_records. The latter is a harmless dry run. First call it as Alice, then restrict tool access using her signed group claim.

The gateway filters discovery and refuses the call before it reaches the tool server. Hiding a tool alone would not be enough.

### Without the control

List the tools as Alice and call the dry-run deletion tool.

**Expected:** Both tools are listed and the dry run is accepted.

In [ ]:
ALICE=$(python3 "$DD/identity.py" token --user alice)
dd_mcp "$ALICE" tools/list
dd_mcp "$ALICE" tools/call '{"name":"delete_records","arguments":{}}'

### Add the control

Permit read_status for authenticated callers and all tools only for the admin group.

**Expected:** The tool policy is Accepted and Attached.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: tool-permissions, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: enterpriseagentgateway.solo.io, kind: EnterpriseAgentgatewayBackend, name: tools}
  backend:
    mcp:
      authorization:
        action: Allow
        policy:
          matchExpressions:
          - 'jwt.group == "admin" || mcp.tool.name == "read_status"'
YAML
dd_wait_policy tool-permissions

### With the control

List Alice's tools, repeat her deletion call, then confirm the admin can still make the dry-run call. Read the refusal log.

**Expected:** Alice sees only read_status and gets Unknown tool for delete_records. Carol's dry run still works.

In [ ]:
ALICE=$(python3 "$DD/identity.py" token --user alice); ADMIN=$(python3 "$DD/identity.py" token --user carol --group admin)
printf 'Reader tools\n'; dd_mcp "$ALICE" tools/list
printf 'Reader call\n'; dd_mcp "$ALICE" tools/call '{"name":"delete_records","arguments":{}}'
printf 'Admin call\n'; dd_mcp "$ADMIN" tools/call '{"name":"delete_records","arguments":{}}'
kubectl --context kind-mesh1 -n dd-gateway logs deploy/dd-gateway --since=15s | grep 'Unknown tool: delete_records'

## 6. Remove personal data before the model

Send a sample email address. First the model receives it unchanged. Add request masking and read what the model actually received on the second call.

Masking is not a refusal. HTTP 200 is expected. The upstream log, not just the response, proves the email was removed. Regex masking does not classify every kind of personal data.

### Without the control

Send the specimen email through the gateway.

**Expected:** The echo model returns alice@example.test unchanged.

In [ ]:
ALICE=$(python3 "$DD/identity.py" token --user alice)
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" -H "Authorization: Bearer $ALICE" \
  -H 'Content-Type: application/json' -d '{"model":"dd-echo","messages":[{"role":"user","content":"Contact alice@example.test for the sample."}]}'

### Add the control

Mask recognised email addresses on the model route. The backend deliberately uses the singular AI provider shape.

**Expected:** The prompt guard is Accepted and Attached.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: mask-personal-data, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, name: model}
  backend:
    ai:
      promptGuard:
        request:
        - regex:
            action: Mask
            builtins: [Email]
YAML
dd_wait_policy mask-personal-data

### With the control

Send the same prompt. Read the last received_prompt from the model's own log, then the gateway's guardrail counter.

**Expected:** The model receives <EMAIL_ADDRESS>, with HTTP 200. The Mask check counter increases.

In [ ]:
ALICE=$(python3 "$DD/identity.py" token --user alice)
kubectl --context kind-mesh1 -n dd-agents exec unmanaged-agent -- curl -sS --max-time 8 -w '\nHTTP %{http_code}\n' "$DD_URL/v1/chat/completions" -H "Authorization: Bearer $ALICE" \
  -H 'Content-Type: application/json' -d '{"model":"dd-echo","messages":[{"role":"user","content":"Contact alice@example.test for the sample."}]}'
printf 'Upstream evidence\n'
kubectl --context kind-mesh1 -n dd-models logs deploy/model --tail=30 | jq -Rsc '[split("\n")[] | fromjson? | select(has("received_prompt"))] | last'
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

## 7. Limit requests per caller

Send six requests. Add a global limit of three model requests per minute per signed subject, then repeat the requests and make a request as someone else.

Each demonstration burst uses fresh Alice and Bob subjects so rerunning the step has a fresh bucket. It sends at most twenty requests. No model spend is involved.

### Without the control

Send the bounded burst with no rate policy.

**Expected:** All Alice requests and Bob's request return 200.

In [ ]:
dd_burst open "${N:-6}"

### Add the control

Apply the per-subject descriptor and policy. Wait for the limiter to accept the current config generation.

**Expected:** The policy is attached and the limiter reports ACCEPTED.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
apiVersion: ratelimit.solo.io/v1alpha1
kind: RateLimitConfig
metadata: {name: dd-per-person, namespace: dd-gateway}
spec:
  raw:
    descriptors:
    - key: dd-sub
      rateLimit: {requestsPerUnit: 3, unit: MINUTE}
    rateLimits:
    - actions:
      - cel: {expression: jwt.sub, key: dd-sub}
      type: REQUEST
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: per-person-rate, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: HTTPRoute, name: model}
  traffic:
    entRateLimit:
      global:
        rateLimitConfigRefs: [{name: dd-per-person}]
        backendRef:
          name: dd-rate-limiter
          namespace: agentgateway-system
          port: 8083
YAML
dd_wait_policy per-person-rate
dd_wait_rate

### With the control

Repeat the burst and read the gateway's 429 evidence.

**Expected:** Alice gets 429 after her allowance. Bob still gets 200.

In [ ]:
dd_burst limited "${N:-6}"
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

## 8. Read the recorded outcomes by caller

The platform has already recorded refusals. Add caller identity to the gateway metrics and generate another bounded burst so the new labels have something to show.

Detection does not refuse requests. The live board combines gateway counters, the waypoint's RBAC counter and retained ztunnel logs. Mask checks and HTTP refusals are labelled separately.

### Before adding caller labels

Read the gateway's own metrics. Status and reason are already present.

**Expected:** 401, 429 and Mask evidence are visible without changing enforcement.

In [ ]:
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

### Add caller labels

Attach a metrics attribute based on the validated JWT subject.

**Expected:** The identity-metrics policy is Accepted and Attached.

In [ ]:
kubectl --context kind-mesh1 apply -f - <<'YAML'
# These are bounded demonstration identities, not a production label-cardinality plan.
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: {name: identity-metrics, namespace: dd-gateway}
spec:
  targetRefs:
  - {group: gateway.networking.k8s.io, kind: Gateway, name: dd-gateway}
  frontend:
    metrics:
      attributes:
        add:
        - {name: caller, expression: jwt.sub}
YAML
dd_wait_policy identity-metrics

### With caller labels

Generate fresh rate refusals and read the metrics again. Watch Detection become Observing on the App tab.

**Expected:** The 429 samples carry Alice's subject, while Bob succeeds. Earlier unlabelled samples may remain until the gateway restarts.

In [ ]:
dd_burst limited 6
POD=$(kubectl --context kind-mesh1 -n dd-gateway get pod -l gateway.networking.k8s.io/gateway-name=dd-gateway -o jsonpath='{.items[0].metadata.name}')
kubectl --context kind-mesh1 get --raw "/api/v1/namespaces/dd-gateway/pods/$POD:15020/proxy/metrics" | grep -E '^agentgateway_(requests_total|guardrail_checks_total)'

## Reset

Remove only this lab's footprint. The console also checks that the resources are gone.

In [ ]:
kubectl --context kind-mesh1 delete namespace dd-agents dd-models dd-tools dd-gateway --ignore-not-found --wait=true --timeout=180s
kubectl --context kind-mesh1 -n agentgateway-system delete service dd-rate-limiter --ignore-not-found
rm -f "$DD_STATE/private.pem" "$DD_STATE/07-caller-identity.json"